## Transformer-Based Emotion Classification

This notebook fine-tunes a pretrained Transformer model (DistilBERT)
for emotion classification on tweets.

Transformer models rely on self-attention and contextual embeddings,
allowing them to capture global dependencies more effectively than
RNN-based architectures such as LSTM and BiLSTM.


In [1]:
import os
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)


/Users/chiranthanshivakumar/NLP/nlp_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


In [3]:
BASE_PATH = "../dataset"

train_df = pd.read_csv(os.path.join(BASE_PATH, "train_clean.csv"))
test_df  = pd.read_csv(os.path.join(BASE_PATH, "test_clean.csv"))

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


Train shape: (16000, 2)
Test shape: (2000, 2)


,clean_text,label
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
label_encoder = LabelEncoder()

train_df["label_id"] = label_encoder.fit_transform(train_df["label"])
test_df["label_id"]  = label_encoder.transform(test_df["label"])

NUM_LABELS = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)


Classes: ['anger' 'fear' 'joy' 'love' 'sadness' 'surprise']


In [5]:
train_dataset = Dataset.from_pandas(
    train_df[["clean_text", "label_id"]]
    .rename(columns={"clean_text": "text", "label_id": "labels"})
)

test_dataset = Dataset.from_pandas(
    test_df[["clean_text", "label_id"]]
    .rename(columns={"clean_text": "text", "label_id": "labels"})
)


In [6]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [7]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )


In [8]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)


Map: 100%|█████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:00<00:00, 52849.62 examples/s]


In [9]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


In [10]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)


Loading weights: 100%|██| 100/100 [00:00<00:00, 1028.14it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted")
    }


In [13]:
training_args = TrainingArguments(
    output_dir="./transformer_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",  
    report_to="none",
    dataloader_pin_memory=False
)

In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,   
    compute_metrics=compute_metrics
)

In [15]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.248591,0.196093,0.924000,0.924082
2,0.155897,0.184777,0.926000,0.925515
3,0.096257,0.180543,0.927000,0.926163


Writing model shards: 100%|████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.45it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=3000, training_loss=0.2476715176900228, metrics={'train_runtime': 822.7628, 'train_samples_per_second': 58.34, 'train_steps_per_second': 3.646, 'total_flos': 584773507144512.0, 'train_loss': 0.2476715176900228, 'epoch': 3.0})

In [16]:
eval_results = trainer.evaluate()
print(eval_results)


{'eval_loss': 0.1805427521467209, 'eval_accuracy': 0.927, 'eval_f1_weighted': 0.9261629917014098, 'eval_runtime': 7.8559, 'eval_samples_per_second': 254.585, 'eval_steps_per_second': 15.912, 'epoch': 3.0}


In [17]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
)


              precision    recall  f1-score   support

       anger       0.92      0.92      0.92       275
        fear       0.87      0.91      0.89       224
         joy       0.95      0.95      0.95       695
        love       0.86      0.83      0.84       159
     sadness       0.96      0.97      0.96       581
    surprise       0.77      0.62      0.69        66

    accuracy                           0.93      2000
   macro avg       0.89      0.87      0.88      2000
weighted avg       0.93      0.93      0.93      2000



In [18]:
SAVE_PATH = "../models/transformer_emotion_model"
os.makedirs(SAVE_PATH, exist_ok=True)

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

assert os.path.exists(SAVE_PATH)
print("Transformer model saved successfully.")


Writing model shards: 100%|████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.91it/s]

Transformer model saved successfully.


### Conclusion

The Transformer-based model achieves the strongest overall performance.
Its self-attention mechanism captures global contextual dependencies
more effectively than LSTM and BiLSTM models, leading to higher accuracy
and weighted F1-score.


In [19]:
assert os.path.exists(SAVE_PATH)
print("Notebook 5 executed and verified successfully.")


Notebook 5 executed and verified successfully.
